# XLM-RoBERTa Fine-Tuning & Evaluation for SemRel2024
**Group 11 — Meinka Singh**

This notebook fine-tunes XLM-RoBERTa on English SemRel2024 data and evaluates zero-shot cross-lingual transfer to Afrikaans.

> ⚠️ Before running: Go to **Runtime → Change runtime type → T4 GPU**

## Cell 1 — Install Dependencies
Run this first, then **restart the runtime** when prompted.

In [1]:
!pip install transformers datasets scipy sentence-transformers accelerate -q

## Cell 2 — Imports

In [11]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from scipy.stats import spearmanr
import torch.nn as nn

# Quick check: are we on GPU?
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# If this prints 'cpu', go to Runtime > Change runtime type > GPU

Using device: cuda


## Cell 3 — Load SemRel2024 Dataset

**Why:** SemRel2024 has sentence pairs scored 0–1 for semantic relatedness. Afrikaans has **no training split** — only dev (375) and test (375). English has training (5,500), dev (250), test (2,600). We train on English, then test on Afrikaans — this is **zero-shot cross-lingual transfer**.

In [3]:
print("Loading SemRel2024 dataset...")

eng = load_dataset("SemRel/SemRel2024", "eng")
afr = load_dataset("SemRel/SemRel2024", "afr")

print("\nEnglish splits:", eng)
print("\nAfrikaans splits:", afr)

# Peek at one example so you understand the structure
print("\nExample English training instance:")
print(eng["train"][0])
# You'll see: sentence1, sentence2, label (float between 0 and 1)

Loading SemRel2024 dataset...

English splits: DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 5500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2600
    })
    dev: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 250
    })
})

Afrikaans splits: DatasetDict({
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 375
    })
    dev: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 375
    })
})

Example English training instance:
{'sentence1': 'It that happens, just pull the plug.', 'sentence2': 'if that ever happens, just pull the plug.', 'label': 1.0}


## Cell 4 — Tokenise the Data

**Why:** Transformers need text converted to token IDs. We concatenate sentence1 and sentence2 with a `[SEP]` separator — XLM-RoBERTa then learns to compare them jointly. Max length 128 is sufficient for this dataset and keeps Colab memory manageable.

In [4]:
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# Apply tokenisation to all splits
eng_tokenized = eng.map(tokenize, batched=True)
afr_tokenized = afr.map(tokenize, batched=True)

# Rename 'label' -> 'labels' (HuggingFace Trainer expects 'labels')
eng_tokenized = eng_tokenized.rename_column("label", "labels")
afr_tokenized = afr_tokenized.rename_column("label", "labels")

# Set format so PyTorch tensors are returned
cols = ["input_ids", "attention_mask", "labels"]
eng_tokenized.set_format("torch", columns=cols)
afr_tokenized.set_format("torch", columns=cols)

print("Tokenisation complete.")
print("English train size:", len(eng_tokenized["train"]))
print("Afrikaans test size:", len(afr_tokenized["test"]))

Map:   0%|          | 0/2600 [00:00<?, ? examples/s]

Tokenisation complete.
English train size: 5500
Afrikaans test size: 375


## Cell 5 — Load Model with Regression Head

**Why:** `num_labels=1` tells HuggingFace to attach a single linear output neuron on top of XLM-RoBERTa's `[CLS]` representation. This outputs a raw score compared to the gold label using **MSE loss** — exactly as described in the proposal.

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,           # Regression, not classification
    problem_type="regression",
)
model.to(device)
print(f"Model loaded: {MODEL_NAME}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: xlm-roberta-base


## Cell 6 — Define Evaluation Metric (Spearman Correlation)

**Why:** Spearman's rank correlation is the official SemEval-2024 metric. It measures how well the model's *ranking* of sentence pairs matches the gold ranking — more meaningful than raw score accuracy for this task.

In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()          # shape: (N,)
    correlation, p_value = spearmanr(predictions, labels)
    return {
        "spearman": round(float(correlation), 4),
        "p_value": round(float(p_value), 6),
    }

print("Metric function defined.")

Metric function defined.


## Cell 7 — Training Configuration

**Why each setting:**
- `num_train_epochs=3`: standard starting point for fine-tuning
- `per_device_train_batch_size=16`: fits in Colab GPU memory
- `learning_rate=2e-5`: well-established for transformer fine-tuning
- `load_best_model_at_end`: saves the checkpoint with the best Spearman score on the English dev set

In [7]:
training_args = TrainingArguments(
    output_dir="./xlmr_semrel_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",            # Renamed from evaluation_strategy in newer versions
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # Mixed precision on GPU (faster)
    report_to="none",                 # Disable wandb/tensorboard
)

print("Training arguments configured.")

Training arguments configured.


## Cell 7b — Fix torchvision conflict

Run this cell, then **restart the runtime** (Runtime → Restart session), then run all cells from Cell 2 onwards again.

In [13]:
import torchvision.io as tvio
import unittest.mock

# Patch the broken VideoReader so datasets' torch formatter doesn't crash
tvio.VideoReader = type("VideoReader", (), {})

## Cell 8 — Fine-Tune on English Training Data

**Why:** We train on English because Afrikaans has no training data. XLM-RoBERTa's shared multilingual vocabulary lets it transfer what it learns about semantic relatedness in English to Afrikaans at test time.

⏱️ *This will take ~20–30 minutes on a free Colab GPU.*

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=eng_tokenized["train"],
    eval_dataset=eng_tokenized["dev"],    # English dev split
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning on English data...")
trainer.train()
print("\nFine-tuning complete.")

Starting fine-tuning on English data...


Epoch,Training Loss,Validation Loss,Spearman,P Value
1,0.030803,0.023970,0.761800,0.000000
2,0.022615,0.023742,0.788800,0.000000
3,0.017177,0.023462,0.794400,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Fine-tuning complete.


## Cell 9 — Evaluate on English Test Set

**Why:** This gives us the in-language performance ceiling. We expect this to be the highest score since the model was trained on English.

In [15]:
print("--- English Test Set Evaluation ---")
eng_results = trainer.evaluate(eng_tokenized["test"])
print(f"Spearman (English test): {eng_results['eval_spearman']}")

--- English Test Set Evaluation ---


Spearman (English test): 0.8222


## Cell 10 — Evaluate on Afrikaans Test Set (Zero-Shot Transfer)

**Why:** This is the key result. No Afrikaans training data was used — the model transfers its English fine-tuning to Afrikaans via XLM-RoBERTa's shared multilingual representations. The gap between English and Afrikaans scores = **cross-lingual performance drop**.

In [16]:
print("--- Afrikaans Test Set Evaluation (zero-shot) ---")
afr_results = trainer.evaluate(afr_tokenized["test"])
print(f"Spearman (Afrikaans test): {afr_results['eval_spearman']}")

# Cross-lingual performance drop
drop = eng_results["eval_spearman"] - afr_results["eval_spearman"]
print(f"\nCross-lingual performance drop: {drop:.4f}")
print("(Compare this against AfriBERTa and Sentence-BERT in the final paper)")

--- Afrikaans Test Set Evaluation (zero-shot) ---


Spearman (Afrikaans test): 0.8036

Cross-lingual performance drop: 0.0186
(Compare this against AfriBERTa and Sentence-BERT in the final paper)


## Cell 11 — Cosine Similarity Baseline (Unfine-Tuned Embeddings)

**Why:** Your proposal mentions raw cosine similarity scores from unfine-tuned embeddings as an internal baseline. This shows how much fine-tuning actually helped by comparing to vanilla XLM-RoBERTa `[CLS]` vectors.

In [17]:
print("--- Computing cosine similarity baseline (no fine-tuning) ---")

base_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

def get_cls_embedding(sentences, batch_size=32):
    """Extract [CLS] token embeddings for a list of sentences."""
    all_embeddings = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True,
                            max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            output = base_model(**encoded)
        cls_vecs = output.last_hidden_state[:, 0, :]   # [CLS] token
        all_embeddings.append(cls_vecs.cpu())
    return torch.cat(all_embeddings, dim=0)

def cosine_similarity_scores(s1_embeddings, s2_embeddings):
    """Compute cosine similarity for each pair."""
    cos = nn.CosineSimilarity(dim=1)
    return cos(s1_embeddings, s2_embeddings).numpy()

# Run on Afrikaans test set
afr_test = afr["test"]
s1_emb = get_cls_embedding(afr_test["sentence1"])
s2_emb = get_cls_embedding(afr_test["sentence2"])
cos_scores = cosine_similarity_scores(s1_emb, s2_emb)
gold_labels = np.array(afr_test["label"])

baseline_spearman, _ = spearmanr(cos_scores, gold_labels)
print(f"Cosine baseline Spearman (Afrikaans, no fine-tuning): {baseline_spearman:.4f}")
print(f"Fine-tuned Spearman (Afrikaans):                      {afr_results['eval_spearman']}")
print(f"Improvement from fine-tuning:                         {afr_results['eval_spearman'] - baseline_spearman:.4f}")

--- Computing cosine similarity baseline (no fine-tuning) ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cosine baseline Spearman (Afrikaans, no fine-tuning): 0.4036
Fine-tuned Spearman (Afrikaans):                      0.8036
Improvement from fine-tuning:                         0.4000


## Cell 12 — Results Summary

Collect all numbers in one place for your final report and for comparison with your groupmates.

In [18]:
results_summary = {
    "model": "XLM-RoBERTa-base",
    "english_test_spearman": eng_results["eval_spearman"],
    "afrikaans_test_spearman": afr_results["eval_spearman"],
    "cross_lingual_drop": round(drop, 4),
    "cosine_baseline_afrikaans": round(baseline_spearman, 4),
    "fine_tuning_gain": round(afr_results["eval_spearman"] - baseline_spearman, 4),
}

print("========== RESULTS SUMMARY ==========")
for k, v in results_summary.items():
    print(f"  {k}: {v}")
print("=====================================")

# Save to a text file
with open("xlmr_results.txt", "w") as f:
    for k, v in results_summary.items():
        f.write(f"{k}: {v}\n")
print("\nResults saved to xlmr_results.txt")

========== RESULTS SUMMARY ==========
  model: XLM-RoBERTa-base
  english_test_spearman: 0.8222
  afrikaans_test_spearman: 0.8036
  cross_lingual_drop: 0.0186
  cosine_baseline_afrikaans: 0.4036
  fine_tuning_gain: 0.4

Results saved to xlmr_results.txt


## Cell 13 — Save the Fine-Tuned Model

**Why:** Colab runtimes disconnect and you'll lose everything. Saving to Google Drive means you keep your trained model.

Uncomment the Drive lines to save there.

In [20]:
# Optional: mount Google Drive to avoid losing the model on disconnect
from google.colab import drive
drive.mount('/content/drive')
save_path = "/content/drive/MyDrive/xlmr_semrel_final"

save_path = "./xlmr_semrel_final"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./xlmr_semrel_final
